In [1]:
import joblib
import pandas as pd
import numpy as np

# Load trained model
model = joblib.load("catboost_model.pkl")

# Load feature names
feature_columns = joblib.load("feature_columns.pkl")

print("Model loaded successfully.")
print("Total Features:", len(feature_columns))

Model loaded successfully.
Total Features: 264


In [2]:
sample_house = {
    "MSSubClass": 60,
    "MSZoning": "RL",
    "LotFrontage": 65,
    "LotArea": 8450,
    "Street": "Pave",
    "Alley": "None",
    "LotShape": "Reg",
    "LandContour": "Lvl",
    "Utilities": "AllPub",
    "LotConfig": "Inside",
    "LandSlope": "Gtl",
    "Neighborhood": "CollgCr",
    "Condition1": "Norm",
    "Condition2": "Norm",
    "BldgType": "1Fam",
    "HouseStyle": "2Story",
    "OverallQual": 7,
    "OverallCond": 5,
    "YearBuilt": 2003,
    "YearRemodAdd": 2003,
    "RoofStyle": "Gable",
    "RoofMatl": "CompShg",
    "Exterior1st": "VinylSd",
    "Exterior2nd": "VinylSd",
    "MasVnrType": "BrkFace",
    "MasVnrArea": 196,
    "ExterQual": "Gd",
    "ExterCond": "TA",
    "Foundation": "PConc",
    "BsmtQual": "Gd",
    "BsmtCond": "TA",
    "BsmtExposure": "No",
    "BsmtFinType1": "GLQ",
    "BsmtFinType2": "Unf",
    "BsmtFinSF1": 706,
    "BsmtFinSF2": 0,
    "BsmtUnfSF": 150,
    "TotalBsmtSF": 856,
    "Heating": "GasA",
    "HeatingQC": "Ex",
    "CentralAir": "Y",
    "Electrical": "SBrkr",
    "1stFlrSF": 856,
    "2ndFlrSF": 854,
    "LowQualFinSF": 0,
    "GrLivArea": 1710,
    "BsmtFullBath": 1,
    "BsmtHalfBath": 0,
    "FullBath": 2,
    "HalfBath": 1,
    "BedroomAbvGr": 3,
    "KitchenAbvGr": 1,
    "KitchenQual": "Gd",
    "TotRmsAbvGrd": 8,
    "Functional": "Typ",
    "Fireplaces": 0,
    "FireplaceQu": "None",
    "GarageType": "Attchd",
    "GarageYrBlt": 2003,
    "GarageFinish": "RFn",
    "GarageCars": 2,
    "GarageArea": 548,
    "GarageQual": "TA",
    "GarageCond": "TA",
    "PavedDrive": "Y",
    "WoodDeckSF": 0,
    "OpenPorchSF": 61,
    "EnclosedPorch": 0,
    "3SsnPorch": 0,
    "ScreenPorch": 0,
    "PoolArea": 0,
    "PoolQC": "None",
    "Fence": "None",
    "MiscFeature": "None",
    "MiscVal": 0,
    "MoSold": 2,
    "YrSold": 2008,
    "SaleType": "WD",
    "SaleCondition": "Normal"
}

input_df = pd.DataFrame([sample_house])

input_df.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,60,RL,65,8450,Pave,None,Reg,Lvl,AllPub,Inside,...,0,0,None,None,None,0,2,2008,WD,Normal


In [3]:
# 1. Total Square Footage
input_df["TotalSF"] = (
    input_df["TotalBsmtSF"] +
    input_df["1stFlrSF"] +
    input_df["2ndFlrSF"]
)

# 2. Total Bathrooms
input_df["TotalBathrooms"] = (
    input_df["FullBath"] +
    0.5 * input_df["HalfBath"] +
    input_df["BsmtFullBath"] +
    0.5 * input_df["BsmtHalfBath"]
)

# 3. House Age
input_df["HouseAge"] = (
    input_df["YrSold"] -
    input_df["YearBuilt"]
)

# 4. Remodel Age
input_df["RemodelAge"] = (
    input_df["YrSold"] -
    input_df["YearRemodAdd"]
)

# 5. Living Space Efficiency
input_df["LivingSpaceEfficiency"] = (
    input_df["GrLivArea"] /
    input_df["LotArea"]
)

# 6. Total Porch Area
input_df["TotalPorchSF"] = (
    input_df["OpenPorchSF"] +
    input_df["EnclosedPorch"] +
    input_df["3SsnPorch"] +
    input_df["ScreenPorch"]
)

print("Total Features:", input_df.shape[1])

Total Features: 85


In [4]:
input_df = pd.get_dummies(input_df)

print(input_df.shape)

(1, 85)


In [5]:
input_df = pd.get_dummies(input_df)

input_df = input_df.reindex(columns=feature_columns, fill_value=0)

print(input_df.shape)

(1, 264)


In [6]:
prediction = model.predict(input_df)

print("Predicted House Price: ₹{:,.2f}".format(prediction[0]))

Predicted House Price: ₹366,418.77


In [7]:
pip install shap

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import shap

explainer = shap.TreeExplainer(model)

shap_values = explainer.shap_values(input_df)

shap_importance = pd.DataFrame({
    "Feature": input_df.columns,
    "SHAP Value": shap_values[0]
})

shap_importance["Absolute SHAP"] = shap_importance["SHAP Value"].abs()

shap_importance = shap_importance.sort_values(
    by="Absolute SHAP",
    ascending=False
)

shap_importance.head(10)

C:\Users\SURYA\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Feature,SHAP Value,Absolute SHAP
36,TotalSF,59658.590953,59658.590953
8,BsmtFinSF1,31036.136182,31036.136182
13,2ndFlrSF,22566.342257,22566.342257
15,GrLivArea,22397.558208,22397.558208
12,1stFlrSF,16942.183549,16942.183549
2,LotArea,15024.000802,15024.000802
37,TotalBathrooms,11666.999924,11666.999924
23,Fireplaces,-6794.570673,6794.570673
7,MasVnrArea,6336.155161,6336.155161
21,KitchenAbvGr,-4954.258877,4954.258877


In [14]:
import joblib
import pandas as pd

model = joblib.load("catboost_model.pkl")

input_data = pd.DataFrame([{
    "OverallQual": 7,
    "GrLivArea": 1710,
    "TotalSF": 2566,
    "GarageCars": 2,
    "TotalBathrooms": 3.5,
    "HouseAge": 15,
    "LotArea": 8450
}])

print(model.predict(input_data))

CatBoostError: catboost/libs/data/model_dataset_compatibility.cpp:81: At position 0 should be feature with name MSSubClass (found OverallQual).

In [16]:
loaded = joblib.load("catboost_model_2.pkl")

print(loaded.predict(input_df))

[12.96259681]


NameError: name 'house_data' is not defined